<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/10-engineering-practices/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10: Exercises

Twelve exercises: four Easy, five Medium and three Hard. They're the small, reusable pieces that real ML projects are built from: reading dependency files, validating configs, logging, timing, retrying, splitting data reproducibly, caching, and saving results safely. One of them asks you to write tests instead of code.

The checks for file-related exercises work in temporary folders, so they don't leave anything in your working directory.

**Run the setup cell first.**

Rules:

- The standard library and NumPy only.
- Solutions are in `03_solutions.ipynb`.

In [ ]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import contextlib
import hashlib
import io
import json
import logging
import math
import platform
import re
import sys
import tempfile
import time
import tracemalloc
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _tmpdir():
    return Path(tempfile.mkdtemp())


def check_parse_requirements(f):
    text = """# core
numpy>=1.26
pandas == 2.2.3   # pinned for the paper
Scikit_Learn

-r dev-requirements.txt
uvicorn[standard]>=0.30 ; python_version >= "3.10"
torch!=2.1.0,<3
"""
    expected = {"numpy": ">=1.26", "pandas": "==2.2.3", "scikit-learn": None,
                "uvicorn": ">=0.30", "torch": "!=2.1.0,<3"}
    _scenarios("parse_requirements", [
        ("a typical file", lambda: f(text), expected,
         "strip comments and markers first, skip lines starting with '-', normalise names"),
        ("empty text", lambda: f(""), {}, None),
        ("only comments and blanks", lambda: f("# nothing\n\n   \n"), {}, None),
        ("compatible release ~=", lambda: f("matplotlib~=3.8"), {"matplotlib": "~=3.8"}, None),
        ("duplicate package raises", lambda: f("numpy>=1.26\nNumPy==2.0"), Raises(ValueError),
         "normalise the name before checking for duplicates"),
    ])


def check_make_config(f):
    def defaults_unchanged():
        f({"lr": 0.5, "epochs": 3})
        return DEFAULTS == {"lr": 0.1, "epochs": 100, "batch_size": 32, "seed": 0}

    def overrides_unchanged():
        o = {"lr": 1}
        f(o)
        return o

    _scenarios("make_config", [
        ("no overrides gives the defaults", lambda: f(), {"lr": 0.1, "epochs": 100, "batch_size": 32, "seed": 0}, None),
        ("overrides are applied", lambda: f({"epochs": 5, "seed": 3}), {"lr": 0.1, "epochs": 5, "batch_size": 32, "seed": 3}, None),
        ("an integer lr is stored as a float", lambda: type(f({"lr": 1})["lr"]), float, "config['lr'] = float(lr)"),
        ("a typo in a key raises KeyError", lambda: f({"learning_rate": 0.5}), Raises(KeyError),
         "compare the override keys with DEFAULTS"),
        ("negative lr raises", lambda: f({"lr": -0.1}), Raises(ValueError), None),
        ("lr as a string raises", lambda: f({"lr": "0.1"}), Raises(ValueError), None),
        ("epochs = 0 raises", lambda: f({"epochs": 0}), Raises(ValueError), None),
        ("epochs as a float raises", lambda: f({"epochs": 10.0}), Raises(ValueError), None),
        ("epochs = True raises", lambda: f({"epochs": True}), Raises(ValueError),
         "isinstance(True, int) is True: exclude bools explicitly"),
        ("negative seed raises", lambda: f({"seed": -1}), Raises(ValueError), None),
        ("DEFAULTS is not modified", defaults_unchanged, True, "build a new dict: {**DEFAULTS, **overrides}"),
        ("the overrides dict is not modified", overrides_unchanged, {"lr": 1}, None),
    ])


def check_setup_logger(f):
    def capture(name, calls=1, level="INFO", messages=(("info", "hello"),)):
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            for _ in range(calls):
                log = f(name, level=level)
            for method, text in messages:
                getattr(log, method)(text)
        return buf.getvalue()

    def with_file():
        path = _tmpdir() / "run.log"
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            log = f("check.file", log_file=path)
            log.info("saved")
            n_handlers = len(log.handlers)
        for h in log.handlers:
            h.flush()
        return (n_handlers, path.read_text())

    _scenarios("setup_logger", [
        ("returns a Logger", lambda: isinstance(f("check.type"), logging.Logger), True, None),
        ("format is 'LEVEL name: message'", lambda: capture("check.fmt"), "INFO check.fmt: hello\n",
         "logging.Formatter('%(levelname)s %(name)s: %(message)s') on a StreamHandler(sys.stdout)"),
        ("calling it three times doesn't repeat messages", lambda: capture("check.dup", calls=3),
         "INFO check.dup: hello\n", "remove existing handlers before adding new ones"),
        ("one handler after repeated calls", lambda: [len(f("check.count").handlers) for _ in range(3)], [1, 1, 1], None),
        ("the level hides lower messages", lambda: capture("check.level", level="WARNING",
                                                           messages=[("info", "quiet"), ("warning", "loud")]),
         "WARNING check.level: loud\n", "logger.setLevel(getattr(logging, level.upper()))"),
        ("doesn't propagate to the root logger", lambda: f("check.prop").propagate, False, None),
        ("log_file adds a second handler that writes the same format", with_file,
         (2, "INFO check.file: saved\n"), "add a logging.FileHandler(log_file) with the same formatter"),
    ])


def check_file_sha256(f):
    folder = _tmpdir()
    small = folder / "small.txt"
    small.write_bytes(b"penguins\n")
    big = folder / "big.bin"
    big.write_bytes(bytes(range(256)) * 1000)
    empty = folder / "empty.bin"
    empty.write_bytes(b"")
    _scenarios("file_sha256", [
        ("a small file", lambda: f(small), hashlib.sha256(b"penguins\n").hexdigest(), None),
        ("chunks give the same hash as reading at once", lambda: f(big, chunk_size=7),
         hashlib.sha256(big.read_bytes()).hexdigest(), "digest.update(chunk) for each chunk"),
        ("an empty file", lambda: f(empty), hashlib.sha256(b"").hexdigest(), None),
        ("a path given as a string", lambda: f(str(small)), hashlib.sha256(b"penguins\n").hexdigest(), None),
        ("returns 64 hex characters", lambda: len(f(small)), 64, "return digest.hexdigest()"),
    ])


def check_Timer(f):
    def basic():
        with f() as t:
            time.sleep(0.05)
        return 0.04 < t.elapsed < 1.0

    def before():
        return f().elapsed

    def enter_returns_self():
        timer = f()
        with timer as t:
            pass
        return t is timer

    def with_error():
        timer = f()
        try:
            with timer:
                time.sleep(0.02)
                raise KeyError("boom")
        except KeyError:
            return timer.elapsed is not None and timer.elapsed > 0.01
        return "the exception was swallowed"

    def reuse():
        timer = f()
        with timer:
            time.sleep(0.05)
        with timer:
            pass
        return timer.elapsed < 0.04

    def logs():
        log = logging.getLogger("check.timer")
        log.handlers.clear()
        log.propagate = False
        log.setLevel(logging.INFO)
        buf = io.StringIO()
        handler = logging.StreamHandler(buf)
        log.addHandler(handler)
        with f("loading", logger=log):
            pass
        log.removeHandler(handler)
        return bool(re.fullmatch(r"loading took \d+\.\d{3} s\n", buf.getvalue()))

    _scenarios("Timer", [
        ("elapsed is None before use", before, None, None),
        ("measures a 0.05 s sleep", basic, True, "time.perf_counter() in __enter__ and __exit__"),
        ("__enter__ returns the timer", enter_returns_self, True, "return self from __enter__"),
        ("records the time and lets the exception through", with_error, True,
         "__exit__ must return False (or None), never True"),
        ("each use measures afresh", reuse, True, "restart the clock in __enter__"),
        ("logs '<label> took <seconds> s'", logs, True, 'logger.info("%s took %.3f s", label, elapsed)'),
    ])


def check_retry(f):
    def flaky(fail_times, exc=ConnectionError):
        calls = []

        def download():
            """Fetch something."""
            calls.append(1)
            if len(calls) <= fail_times:
                raise exc("network down")
            return "data"
        return download, calls

    def run(fail_times, times=3, exceptions=(ConnectionError,), exc=ConnectionError):
        func, calls = flaky(fail_times, exc)
        try:
            result = f(times=times, exceptions=exceptions)(func)()
        except Exception as e:
            result = type(e).__name__
        return (result, len(calls))

    def sleeps_with_delay():
        recorded = []
        real_sleep = time.sleep
        time.sleep = recorded.append
        try:
            func, _ = flaky(5)
            try:
                f(times=3, exceptions=(ConnectionError,), delay=0.5)(func)()
            except ConnectionError:
                pass
        finally:
            time.sleep = real_sleep
        return recorded

    def keeps_name():
        func, _ = flaky(0)
        wrapped = f()(func)
        return (wrapped.__name__, wrapped.__doc__)

    _scenarios("retry", [
        ("succeeds first time: one call", lambda: run(0), ("data", 1), None),
        ("fails twice then succeeds", lambda: run(2), ("data", 3), None),
        ("always fails: re-raises after 3 calls", lambda: run(10), ("ConnectionError", 3), "after the last attempt, raise"),
        ("other exceptions are not retried", lambda: run(10, exc=KeyError), ("KeyError", 1),
         "only catch the exception types you were given"),
        ("times=1 means no retries", lambda: run(10, times=1), ("ConnectionError", 1), None),
        ("sleeps between attempts, not after the last", sleeps_with_delay, [0.5, 0.5],
         "3 attempts have 2 gaps between them"),
        ("keeps the function's name and docstring", keeps_name, ("download", "Fetch something."), "@functools.wraps(func)"),
        ("times=0 raises", lambda: f(times=0), Raises(ValueError), None),
    ])


def check_split_indices(f):
    def covers():
        train, test = f(100, 0.2, seed=1)
        return (len(train), len(test), sorted(np.concatenate([train, test]).tolist()) == list(range(100)))

    def global_state():
        state = np.random.get_state()[1].copy()
        f(50, 0.3, seed=7)
        return bool(np.array_equal(state, np.random.get_state()[1]))

    def same_seed():
        a, b = f(30, 0.3, seed=5), f(30, 0.3, seed=5)
        return bool(np.array_equal(a[0], b[0]) and np.array_equal(a[1], b[1]))

    def different_seeds():
        return not np.array_equal(f(30, 0.3, seed=1)[1], f(30, 0.3, seed=2)[1])

    def sorted_ints():
        train, test = f(40, 0.25, seed=0)
        return (bool(np.all(np.diff(train) > 0) and np.all(np.diff(test) > 0)), train.dtype.kind, test.dtype.kind)

    _scenarios("split_indices", [
        ("sizes, and every index exactly once", covers, (80, 20, True), "one permutation, then slice it"),
        ("rounds the test size", lambda: len(f(10, 0.33, seed=0)[1]), 3, None),
        ("sorted integer arrays", sorted_ints, (True, "i", "i"), None),
        ("same seed, same split", same_seed, True, None),
        ("different seeds, different splits", different_seeds, True, None),
        ("doesn't touch NumPy's global random state", global_state, True,
         "use rng = np.random.default_rng(seed), not np.random.seed"),
        ("test_fraction of 1 raises", lambda: f(10, 1.0, seed=0), Raises(ValueError), None),
        ("an empty test set raises", lambda: f(10, 0.01, seed=0), Raises(ValueError), None),
    ])


def _std_correct(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be 2D")
    std = X.std(axis=0)
    return (X - X.mean(axis=0)) / np.where(std == 0, 1.0, std)


def _std_global(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be 2D")
    s = X.std()
    return (X - X.mean()) / (s if s else 1.0)


def _std_ddof1(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be 2D")
    std = X.std(axis=0, ddof=1)
    return (X - X.mean(axis=0)) / np.where(std == 0, 1.0, std)


def _std_inplace(X):
    if np.ndim(X) != 2:
        raise ValueError("X must be 2D")
    std = X.std(axis=0)
    X -= X.mean(axis=0)
    X /= np.where(std == 0, 1.0, std)
    return X


def _std_nan(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be 2D")
    with np.errstate(divide="ignore", invalid="ignore"):
        return (X - X.mean(axis=0)) / X.std(axis=0)


def _std_any_shape(X):
    X = np.asarray(X, dtype=float)
    std = X.std(axis=0)
    return (X - X.mean(axis=0)) / np.where(std == 0, 1.0, std)


def check_test_standardize(f):
    def outcome(impl):
        try:
            result = f(impl)
        except KeyboardInterrupt:
            raise
        except BaseException:          # pytest's failures derive from BaseException, not Exception
            return "failed"
        return "passed" if result is None else f"returned {result!r}"

    _scenarios("test_standardize", [
        ("your test passes the correct version", lambda: outcome(_std_correct), "passed",
         "your test fails on correct code: check your expected values and tolerances"),
        ("catches: one mean and std for the whole array", lambda: outcome(_std_global), "failed",
         "use test data whose columns have very different means and spreads"),
        ("catches: ddof=1 instead of ddof=0", lambda: outcome(_std_ddof1), "failed",
         "check the std of each column is 1 with a tight tolerance, e.g. pytest.approx(1)"),
        ("catches: the input is modified", lambda: outcome(_std_inplace), "failed",
         "take a copy before the call and compare afterwards"),
        ("catches: a constant column gives NaN", lambda: outcome(_std_nan), "failed",
         "test a column where every value is the same"),
        ("catches: non-2D input is accepted", lambda: outcome(_std_any_shape), "failed",
         "check that 1D input raises ValueError"),
    ])


def check_predict_in_batches(f):
    X = np.arange(23 * 3, dtype=float).reshape(23, 3)

    def row_sums(A):
        return A.sum(axis=1)

    def two_cols(A):
        return np.column_stack([A[:, 0], A[:, 0] * 2])

    def batch_sizes():
        seen = []

        def record(A):
            seen.append(len(A))
            return A[:, 0]
        f(record, X, 5)
        return seen

    _scenarios("predict_in_batches", [
        ("1D output matches predicting all at once", lambda: f(row_sums, X, 5), row_sums(X),
         "np.concatenate(outputs, axis=0)"),
        ("2D output matches predicting all at once", lambda: f(two_cols, X, 4), two_cols(X), None),
        ("batches of at most batch_size rows", batch_sizes, [5, 5, 5, 5, 3], "range(0, len(X), batch_size)"),
        ("batch_size bigger than the data", lambda: f(row_sums, X, 100), row_sums(X), None),
        ("batch_size of 1", lambda: f(row_sums, X, 1), row_sums(X), None),
        ("batch_size 0 raises", lambda: f(row_sums, X, 0), Raises(ValueError), None),
    ])


def check_pairwise_distances(f):
    rng = np.random.default_rng(3)
    A, B = rng.normal(size=(7, 4)), rng.normal(size=(5, 4))
    reference = np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2))

    def self_distances():
        P = rng.normal(loc=1000, size=(50, 3))
        D = f(P, P)
        return (bool(np.all(np.isfinite(D))), bool(np.all(D >= 0)), bool(np.allclose(np.diag(D), 0, atol=1e-4)))

    def memory():
        X, Y = rng.normal(size=(600, 64)), rng.normal(size=(500, 64))
        tracemalloc.start()
        f(X, Y)
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        return peak / 1e6 < 30

    _scenarios("pairwise_distances", [
        ("matches the direct formula", lambda: f(A, B), reference, None),
        ("shape (n, m)", lambda: f(A, B).shape, (7, 5), None),
        ("integer input", lambda: f([[0, 0], [3, 4]], [[0, 0]]), np.array([[0.0], [5.0]]), None),
        ("distances to itself: no NaN, never negative, diagonal close to 0", self_distances, (True, True, True),
         "clip the squared distances at 0 before np.sqrt"),
        ("peak memory under 30 MB for 600 x 500 points in 64 dimensions", memory, True,
         "expand the square: sum(X**2) + sum(Y**2) - 2 * X @ Y.T, no (n, m, d) array"),
    ])


def check_LRUCache(f):
    def basic():
        c = f(2)
        c.put("a", 1)
        c.put("b", 2)
        return (c.get("a"), c.get("b"), c.get("zzz"), c.get("zzz", "none"), len(c))

    def evicts_oldest():
        c = f(2)
        c.put("a", 1)
        c.put("b", 2)
        c.put("c", 3)
        return ("a" in c, "b" in c, "c" in c, len(c))

    def get_counts_as_use():
        c = f(2)
        c.put("a", 1)
        c.put("b", 2)
        c.get("a")
        c.put("c", 3)
        return ("a" in c, "b" in c, "c" in c)

    def put_updates():
        c = f(2)
        c.put("a", 1)
        c.put("b", 2)
        c.put("a", 10)
        c.put("c", 3)
        return (c.get("a"), "b" in c, len(c))

    def contains_is_not_use():
        c = f(2)
        c.put("a", 1)
        c.put("b", 2)
        _ = "a" in c
        c.put("c", 3)
        return ("a" in c, "b" in c)

    def stats():
        c = f(3)
        c.put("x", None)
        results = [c.get("x"), c.get("y"), c.get("x"), c.get("x", "default")]
        return (results, c.hits, c.misses)

    _scenarios("LRUCache", [
        ("put, get, default and len", basic, (1, 2, None, "none", 2), None),
        ("a full cache evicts the least recently used", evicts_oldest, (False, True, True, 2),
         "OrderedDict.popitem(last=False) removes the oldest"),
        ("get counts as a use", get_counts_as_use, (True, False, True), "move_to_end on every get"),
        ("putting an existing key updates it and counts as a use", put_updates, (10, False, 2), None),
        ("'in' does not count as a use", contains_is_not_use, (False, True), None),
        ("hits and misses, even for a stored None", stats, ([None, None, None, None], 3, 1),
         "check 'key in items', not whether the value is None"),
        ("capacity 0 raises", lambda: f(0), Raises(ValueError), None),
    ])


def check_save_run(f):
    def basic():
        folder = _tmpdir() / "runs" / "exp1"
        path = f({"lr": 0.1}, {"accuracy": 0.9}, folder)
        data = json.loads(Path(path).read_text())
        return (Path(path) == folder / "run.json", sorted(data), data["config"], data["metrics"])

    def environment():
        data = json.loads(Path(f({}, {}, _tmpdir())).read_text())
        return data["environment"] == {"python": platform.python_version(), "numpy": np.__version__}

    def timestamp():
        data = json.loads(Path(f({}, {}, _tmpdir())).read_text())
        created = datetime.fromisoformat(data["created"])
        return created.tzinfo is not None and abs((datetime.now(timezone.utc) - created).total_seconds()) < 60

    def numpy_values():
        metrics = {"accuracy": np.float64(0.91), "n": np.int64(81), "ok": np.bool_(True),
                   "per_class": np.array([0.9, 0.95])}
        data = json.loads(Path(f({}, metrics, _tmpdir())).read_text())
        return data["metrics"]

    def refuses_overwrite():
        folder = _tmpdir()
        f({"v": 1}, {}, folder)
        try:
            f({"v": 2}, {}, folder)
        except FileExistsError:
            return json.loads((folder / "run.json").read_text())["config"]
        return "no error: the first result was replaced"

    def overwrite_allowed():
        folder = _tmpdir()
        f({"v": 1}, {}, folder)
        f({"v": 2}, {}, folder, overwrite=True)
        return json.loads((folder / "run.json").read_text())["config"]

    def bad_value():
        folder = _tmpdir() / "bad"
        try:
            f({}, {"seen": {1, 2}}, folder)
        except TypeError:
            return sorted(p.name for p in folder.glob("*")) if folder.exists() else []
        return "no TypeError raised"

    _scenarios("save_run", [
        ("creates folders and writes the four keys", basic,
         (True, ["config", "created", "environment", "metrics"], {"lr": 0.1}, {"accuracy": 0.9}), None),
        ("records Python and NumPy versions", environment, True, None),
        ("'created' is a time-zone-aware ISO timestamp", timestamp, True, "datetime.now(timezone.utc).isoformat()"),
        ("NumPy numbers and arrays are converted", numpy_values,
         {"accuracy": 0.91, "n": 81, "ok": True, "per_class": [0.9, 0.95]},
         "json.dumps(..., default=converter): .item() for np.generic, .tolist() for arrays"),
        ("refuses to replace an existing run", refuses_overwrite, {"v": 1}, "raise FileExistsError if run.json exists"),
        ("overwrite=True replaces it", overwrite_allowed, {"v": 2}, None),
        ("unsaveable value: TypeError and no file left behind", bad_value, [],
         "build the JSON text before creating any file"),
    ])


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `parse_requirements`

Write `parse_requirements(text)` that reads the contents of a `requirements.txt` file and returns a dict mapping each package name to its version specifier.

```
numpy>=1.26          ->  "numpy": ">=1.26"
pandas == 2.2.3      ->  "pandas": "==2.2.3"
Scikit_Learn         ->  "scikit-learn": None
uvicorn[standard]>=0.30 ; python_version >= "3.10"   ->  "uvicorn": ">=0.30"
```

Rules:

- Ignore blank lines, comments (`#` and everything after it, including at the end of a line), and option lines that start with `-` (such as `-r other.txt`).
- Drop anything after `;` (an environment marker) and any extras in square brackets.
- Package names are case-insensitive and treat `_` and `-` as the same, so normalise them to **lowercase with `-`**.
- The specifier is everything from the first `=`, `<`, `>`, `!` or `~`, with spaces removed, or `None` if there isn't one.
- A package listed twice is a mistake: raise `ValueError`.

In [ ]:
def parse_requirements(text: str) -> dict:
    # your code here
    pass

In [ ]:
check_parse_requirements(parse_requirements)

### Exercise 2: `make_config`

Write `make_config(overrides=None)` that starts from `DEFAULTS` (given below), applies the overrides, validates the result, and returns it as a **new** dict.

- An override key that isn't in `DEFAULTS` raises `KeyError` naming the key. (This catches typos like `"learning_rate"`.)
- `lr` must be a positive number. An integer like `1` is fine, but store it as a float.
- `epochs` and `batch_size` must be integers of at least 1. `seed` must be an integer of at least 0.
- Anything else raises `ValueError`. Note that `True` counts as an integer in Python (`isinstance(True, int)` is `True`), but a boolean is never a sensible epoch count, so reject it.
- Don't modify `DEFAULTS` or the `overrides` dict.

In [ ]:
DEFAULTS = {"lr": 0.1, "epochs": 100, "batch_size": 32, "seed": 0}


def make_config(overrides=None) -> dict:
    # your code here
    pass

In [ ]:
check_make_config(make_config)

### Exercise 3: `setup_logger`

Write `setup_logger(name, level="INFO", log_file=None)` that returns a configured `logging.Logger`.

- Set the logger's level from the level **name** (`"DEBUG"`, `"INFO"`, `"WARNING"`, ...).
- Add a handler that writes to `sys.stdout`, with the format `"%(levelname)s %(name)s: %(message)s"`.
- If `log_file` is given, also add a `FileHandler` writing to that file, with the same format.
- Set `propagate = False`, so messages aren't passed on to the root logger as well.
- **Calling it again with the same name must not add more handlers.** Remove (and close) any existing handlers first, so every message appears once.

In [ ]:
import logging
import sys


def setup_logger(name: str, level: str = "INFO", log_file=None) -> logging.Logger:
    # your code here
    pass

In [ ]:
check_setup_logger(setup_logger)

### Exercise 4: `file_sha256`

Write `file_sha256(path, chunk_size=1_048_576)` that returns the SHA-256 hash of a file as a hex string (64 characters).

Read the file in chunks of `chunk_size` bytes, not all at once, so it works on files bigger than memory. Use `hashlib.sha256()` and its `update` method, which can be fed the data piece by piece.

`path` may be a string or a `Path`.

In [ ]:
import hashlib


def file_sha256(path, chunk_size: int = 1_048_576) -> str:
    # your code here
    pass

In [ ]:
check_file_sha256(file_sha256)

## Medium

### Exercise 5: `Timer`

Write a class `Timer` that measures how long a block of code takes, used as a context manager:

```python
with Timer("training") as t:
    train_model()
print(t.elapsed)          # seconds, as a float
```

- `Timer(label=None, logger=None)`. `elapsed` is `None` until a block has finished.
- `__enter__` starts the clock with `time.perf_counter()` and returns the timer itself.
- `__exit__` stores the elapsed time. If a `logger` was given, it logs `"<label> took <seconds> s"` at INFO level, with seconds to 3 decimal places.
- If the block raises an exception, `elapsed` must still be recorded, and the exception must **not** be swallowed.
- The same timer can be used for more than one block; each use measures afresh.

In [ ]:
import time


class Timer:
    # your code here
    pass

In [ ]:
check_Timer(Timer)

### Exercise 6: `retry`

Write a decorator factory `retry(times=3, exceptions=(Exception,), delay=0.0)` for calls that sometimes fail, such as downloading a file.

```python
@retry(times=3, exceptions=(ConnectionError,), delay=1.0)
def download(url): ...
```

- Call the function up to `times` times in total. Return the result of the first call that succeeds.
- Only retry on the given exception types. Any other exception goes straight through on the first attempt.
- If every attempt fails, raise the **last** exception.
- Wait `delay` seconds between attempts (`time.sleep`), but not after the final one.
- Use `functools.wraps`, so the decorated function keeps its name and docstring.
- `times` less than 1 raises `ValueError` (when `retry` is called).

In [ ]:
import functools
import time


def retry(times: int = 3, exceptions=(Exception,), delay: float = 0.0):
    # your code here
    pass

In [ ]:
check_retry(retry)

### Exercise 7: `split_indices`

Write `split_indices(n, test_fraction, seed)` that returns `(train_idx, test_idx)`, two sorted integer arrays that split `range(n)` at random.

- The test set has `round(n * test_fraction)` indices; the training set has the rest. Together they contain every index exactly once.
- The same `seed` always gives the same split. Different seeds should (almost always) give different splits.
- **Don't use or change NumPy's global random state** (`np.random.seed`, `np.random.permutation`, ...). Create a `Generator` from the seed.
- Raise `ValueError` if `test_fraction` isn't strictly between 0 and 1, or if either set would be empty.

In [ ]:
def split_indices(n: int, test_fraction: float, seed: int):
    # your code here
    pass

In [ ]:
check_split_indices(split_indices)

### Exercise 8: `test_standardize`

This time you write the **tests**. Here is a correct function:

```python
def standardize(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be 2D")
    std = X.std(axis=0)
    return (X - X.mean(axis=0)) / np.where(std == 0, 1.0, std)
```

Its specification:

1. Each column of the result has mean 0 and standard deviation 1 (population standard deviation, `ddof=0`), computed **per column**.
2. A constant column becomes all zeros, not `NaN`.
3. The input is not modified.
4. Input that isn't 2D raises `ValueError`.

Write `test_standardize(standardize)`, a function that takes **an implementation** as its argument and checks it with `assert` statements (and `pytest.raises` or `try`/`except` for the error case). It should return nothing if the implementation is correct, and fail with an exception if it isn't.

The check runs your test against the correct version, which must pass, and against **five broken versions**, each of which must fail. It tells you which bug your tests missed.

In [ ]:
import pytest


def test_standardize(standardize):
    # your code here
    pass

In [ ]:
check_test_standardize(test_standardize)

### Exercise 9: `predict_in_batches`

Lesson section 9 showed that fast, vectorised prediction can need a lot of memory. Write `predict_in_batches(predict, X, batch_size)` that calls `predict` on consecutive slices of `X` of at most `batch_size` rows, and combines the results.

- The result must equal `predict(X)` called on all rows at once, for outputs that are 1D (one value per row) or 2D (one row of values per input row).
- `predict` must never receive more than `batch_size` rows.
- The last batch may be smaller.
- `batch_size` less than 1 raises `ValueError`.

In [ ]:
def predict_in_batches(predict, X, batch_size: int):
    # your code here
    pass

In [ ]:
check_predict_in_batches(predict_in_batches)

## Hard

### Exercise 10: `pairwise_distances`

Write `pairwise_distances(X, Y)` that returns the matrix of Euclidean distances between every row of `X` (shape `(n, d)`) and every row of `Y` (shape `(m, d)`), with shape `(n, m)`.

The broadcasting version from the lesson builds an `(n, m, d)` array first. For `n = 600`, `m = 500` and `d = 64` that's about 150 MB just for an intermediate result. **Your version must use far less memory**: the check requires a peak below 30 MB for that size (the output itself is 2.4 MB).

The trick is to expand the square:

$$\|x - y\|^2 = \|x\|^2 + \|y\|^2 - 2\, x \cdot y$$

The `x · y` part for all pairs is a single matrix multiplication. One more detail: rounding can make a squared distance come out as a tiny negative number, like `-1e-13`, when it should be 0, and `np.sqrt` of a negative number is `nan`. Make sure that can't happen.

In [ ]:
def pairwise_distances(X, Y):
    # your code here
    pass

In [ ]:
check_pairwise_distances(pairwise_distances)

### Exercise 11: `LRUCache`

Write a class `LRUCache(capacity)`, a cache that holds at most `capacity` items and, when full, throws away the **least recently used** one.

- `put(key, value)` stores a value. Storing an existing key updates its value and counts as a use.
- `get(key, default=None)` returns the stored value and counts as a use, or returns `default` if the key isn't there.
- When a `put` adds a new key to a full cache, remove the item that was used longest ago.
- `len(cache)` and `key in cache` work. Checking `in` does **not** count as a use.
- Count `hits` and `misses` (attributes) for `get` calls.
- `capacity` less than 1 raises `ValueError`.

`collections.OrderedDict` makes this short: it remembers insertion order and has `move_to_end` and `popitem(last=False)`.

In [ ]:
from collections import OrderedDict


class LRUCache:
    # your code here
    pass

In [ ]:
check_LRUCache(LRUCache)

### Exercise 12: `save_run`

Write `save_run(config, metrics, out_dir, overwrite=False)` that saves an experiment's record to `out_dir/run.json` and returns that path.

The JSON object has four keys:

- `"config"` and `"metrics"`: the dicts passed in.
- `"environment"`: `{"python": ..., "numpy": ...}` with the version strings.
- `"created"`: the current time as an ISO 8601 string **with a time zone** (use `datetime.now(timezone.utc).isoformat()`).

Requirements:

- Create `out_dir` (and any missing parent folders).
- If `run.json` already exists, raise `FileExistsError`, unless `overwrite=True`. Never silently replace a previous result.
- Metrics often contain NumPy values (`np.float64`, `np.int64`, small arrays), which the `json` module refuses. Convert them: NumPy numbers to Python numbers, arrays to lists.
- If something can't be converted, raise `TypeError`, and **don't leave a half-written or empty `run.json` behind**. Hint: build the full JSON text before writing anything.

In [ ]:
import json
import platform
from datetime import datetime, timezone


def save_run(config: dict, metrics: dict, out_dir, overwrite: bool = False) -> Path:
    # your code here
    pass

In [ ]:
check_save_run(save_run)

## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the mini project in `04_mini_project.ipynb`, where you'll turn the penguin classifier from Module 9 into a small, tested, configurable project with a command-line interface.